In [1]:
# Loading libraries

import astropy
import os
import numpy as np
import pandas as pd
import matplotlib
import scipy
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table
from matplotlib import pyplot as plt


In [2]:
alma_regions = Table.read("../alma_input/almaprops_2024_all_20250219.tab", format="ascii.ipac")
print(alma_regions)

    ALMAGAL_ID           ORIG_ID         ... FIT_FLAG OBS_SAMPLE
                                         ...                    
----------------- ---------------------- ... -------- ----------
AG010.1313-0.7687  HIGALBM10.1330-0.7710 ...        -       near
AG010.1392-0.7831  HIGALBM10.1395-0.7833 ...        -       near
AG010.2283-0.2064  HIGALBM10.2282-0.2069 ...        -       near
AG010.3226-0.1599  HIGALBM10.3229-0.1596 ...        -       near
AG010.6185-0.0314  HIGALBM10.6183-0.0314 ...        -        far
AG010.6691-0.2196  HIGALBM10.6693-0.2188 ...        -       near
AG010.6985-0.0994  HIGALBM10.6975-0.0984 ...        -        far
AG010.7516-0.1987  HIGALBM10.7493-0.1984 ...        -       near
AG010.8851+0.1225  HIGALBM10.8852+0.1231 ...        -       near
              ...                    ... ...      ...        ...
AG348.5792-0.9197 HIGALBM348.5789-0.9198 ...        -       near
AG348.5918+0.1471 HIGALBM348.5889+0.1463 ...        -        far
AG348.7017-1.0414 HIGALBM

In [ ]:
spitzer_clumps = Table.read("../output/spitzer_clumps.tbl", format="ascii.ipac")
print(spitzer_clumps)

cntr_01   dist_x     pang_x   ... color45_58 color45_80 color58_80
          arcsec      deg     ...                                 
------- --------- ----------- ... ---------- ---------- ----------
      1  2.867081  -55.508188 ...      0.471      0.367     -0.104
      1  4.406157   86.064855 ...         --         --         --
      1  8.680204  -84.956251 ...      0.342         --         --
      1 11.244409  107.572919 ...      0.204         --         --
      1 12.740404  176.146569 ...      0.662      1.394      0.732
      1 14.110753    2.108431 ...         --         --         --
      1 14.586334   88.756257 ...         --         --         --
      1 14.719587  167.971572 ...         --         --         --
      1 15.372734   23.769197 ...         --         --         --
    ...       ...         ... ...        ...        ...        ...
   1017  58.44517  134.662464 ...         --         --         --
   1017 58.593662  -82.369246 ...         --         --       

In [4]:
alma_clumps = pd.read_fwf('../alma_input/7MTM2TM1_Core_catalogue_out_official_v3_run6Apr23+16Jan24_sn-5_may24_clump_cat.txt',comment='\\',skiprows=[2])
print(alma_clumps)

      ID   CLUMP          X          Y         RA        DEC        GLON  \
0      1  100132  897.03558  807.00440  277.80605  -8.909354   22.706657   
1      2  100132  796.61502  862.99532  277.80701  -8.908826   22.707565   
2      4  100244  468.07833  931.00007  278.45735  -9.216649   22.731143   
3      1  100309  768.08734  459.99554  278.38876  -9.176013   22.735978   
4     12  100309  696.04825  692.00105  278.38951  -9.173629   22.738435   
...   ..     ...        ...        ...        ...        ...         ...   
6343  10  G349.6  419.91840  417.02003  260.75494 -38.231123  349.643980   
6344   4  G349.6  393.95789  354.99335  260.75550 -38.232174  349.643360   
6345   5  G349.6  384.22266  360.00968  260.75571 -38.232089  349.643520   
6346   6  G349.6  374.02368  380.01611  260.75593 -38.231750  349.643900   
6347   7  G349.6  374.95074  387.98336  260.75591 -38.231615  349.644000   

          GLAT      F_PEAK         S/N  ...  Back_onpeak_jybm       Rcore  \
0     0.39

In [13]:
match_table = Table(
    names=['alma_clump','spitzer_clump','alma_ra','alma_dec','spitzer_ra','spitzer_dec'],
    dtype=['str','str','float64','float64','float64','float64']
)

count = 0

for region in alma_regions[:100]: # Just for the first 100 regions

    count +=1
    spitzer_clumps_cut = spitzer_clumps[spitzer_clumps["internal_id_01"] == str(region["INTERNAL_ID"])]
    alma_clumps_cut = alma_clumps[alma_clumps["CLUMP"] == str(region["INTERNAL_ID"])].reset_index(drop=True)

    #print(f"Region {region['INTERNAL_ID']} has {len(spitzer_clumps_cut)} Spitzer clumps and {len(alma_clumps_cut)} ALMA clumps.")
    for alma_index in range(len(alma_clumps_cut)):
        clump_coords = SkyCoord(alma_clumps_cut["RA"][alma_index], alma_clumps_cut["DEC"][alma_index], unit=(u.deg, u.deg), frame="icrs")

        radius = 1*u.arcsec

        #plt.scatter(clump_coords.ra.deg, clump_coords.dec.deg, color="red", label="ALMA Clump")

        for spitzer_clump in spitzer_clumps_cut:
            spitzer_coords = SkyCoord(spitzer_clump["ra"], spitzer_clump["dec"], unit=(u.deg, u.deg), frame="icrs")

            if clump_coords.separation(spitzer_coords) < radius:
                #print(f"ALMA clump {alma_clumps_cut['CLUMP'][alma_index]} is close to Spitzer clump {spitzer_clump['designation']}")
                #plt.scatter(spitzer_coords.ra.deg, spitzer_coords.dec.deg, color="blue", label="Spitzer Clump")

                match_table.add_row([alma_clumps_cut["CLUMP"][alma_index], spitzer_clump["designation"], 
                                     clump_coords.ra.deg, clump_coords.dec.deg,spitzer_clump["ra"], spitzer_clump["dec"]])
        print(f"progress: {count}/{len(alma_clumps)}", end="\r")

    #plt.title(f"Region {region['INTERNAL_ID']}")

    #g = SkyCoord(region["GLON"], region["GLAT"], unit=(u.deg, u.deg), frame="galactic").icrs

    #plt.ylim(g.dec.deg - 0.01, g.dec.deg + 0.01)
    #plt.xlim(g.ra.deg - 0.01, g.ra.deg + 0.01)
    #plt.ylabel("Declination (deg)")
    #plt.xlabel("Right Ascension (deg)")    
    #if alma_index <= 3:
    #    plt.show()
    #print("\n")


In [15]:
print(match_table)

match_table.write("../output/match_table.tbl", format="ascii.ipac", overwrite=True)

alma_clump       spitzer_clump        alma_ra   alma_dec  spitzer_ra spitzer_dec
---------- ------------------------- --------- ---------- ---------- -----------
     47184 SSTGLMC G010.2278-00.2071 272.25151 -20.191408 272.251339  -20.191348
     47184 SSTGLMC G010.2278-00.2071 272.25113 -20.191168 272.251339  -20.191348
     49005 SSTGLMC G010.6702-00.2214 272.49237 -19.811085 272.492445  -19.811121
     50555 SSTGLMC G011.0844-00.5303 272.99201 -19.596941 272.991994  -19.597059
     51554 SSTGLMC G011.3620+00.8027  271.8988 -18.709561 271.898735  -18.709532
     51554 SSTGLMC G011.3640+00.8014 271.90085 -18.708417 271.900995  -18.708429
     52831 SSTGLMC G011.6987-00.3028 273.09338 -18.948836 273.093365  -18.949053
     53795 SSTGLMC G011.9019-00.1433 273.04848 -18.694362 273.048646  -18.694229
     53795 SSTGLMC G011.9040-00.1411 273.04766 -18.691371 273.047742  -18.691359
     53869 SSTGLMC G011.9186-00.6126 273.49223 -18.904526 273.492217  -18.904616
       ...                  